# 02 — Generative Model Design for Cross-Modal AIF
## Visuo-Tactile Occlusion Compensation on a 1-DoF Arm (SO-101 Prototype)

This notebook designs and validates the **cross-modal generative model** at the heart of the research.
Building directly on the T-maze from `01_pymdp_basics.ipynb`, we map the same 4-matrix structure
onto a simplified robot arm task where **visual occlusion forces the agent to rely on tactile contact**.

### Structure
1. From T-maze to arm manipulation — structural analogy
2. State / observation / action space design
3. Generative model: A, B, C, D matrices
4. Precision switching: occlusion → visual A degrades
5. Simulation comparison: no occlusion vs complete occlusion
6. Bridge to SO-101 full system

---
## 1. From T-maze to Arm Manipulation

The T-maze and the arm task share the same AIF structure:

| T-maze concept | Arm task equivalent |
|---|---|
| Agent position (4 locations) | Arm position (5 discrete levels) |
| Reward location (left / right) | **Object location (left / center / right)** |
| Cue observation (visual hint) | **Visual observation** (external camera) |
| Reward observation | **Tactile contact** (servo current / AnySkin) |
| Noise in cue modality | **Precision drop** in visual (occlusion) |
| Move to cue → move to reward | Move arm → **search until contact** |

**Key insight**: when visual is occluded (cue modality disabled), the agent must rely on
tactile contact observations to infer the object location — exactly as the T-maze agent
falls back on exploratory movement when the cue is unreliable.

```
Arm positions (1-D workspace):

  pos0 --- pos1 --- pos2 --- pos3 --- pos4
  start    [L?]    [C?]    [R?]

Object is hidden at one of three locations (left=pos1, center=pos2, right=pos3).
The agent starts at pos0 and must reach and contact the object.
```

In [ ]:
import numpy as np
import copy
import matplotlib.pyplot as plt
import matplotlib
matplotlib.rcParams['font.family'] = 'DejaVu Sans'
matplotlib.rcParams['axes.unicode_minus'] = False

from pymdp.legacy import utils
from pymdp.legacy.agent import Agent

print('Imports OK')
print('numpy:', np.__version__)

---
## 2. State / Observation / Action Space

### Hidden states (2 factors)
| Factor | Variable | Levels | Meaning |
|---|---|---|---|
| 0 | Arm position | 5 | Discrete joint angle: pos0 … pos4 |
| 1 | Object location | 3 | Where the object is hidden: left(pos1) / center(pos2) / right(pos3) |

### Observations (2 modalities)
| Modality | Sensor | Levels | Reliability |
|---|---|---|---|
| 0 | **Visual** (external camera) | 4 | High when unoccluded; degrades with Π_visual ↓ |
| 1 | **Tactile** (servo current / AnySkin) | 2 | Always reliable (Π_tactile = const) |

### Actions
| Index | Action | Effect |
|---|---|---|
| 0 | Move left | arm_pos − 1 (boundary: ≥ 0) |
| 1 | Stay | arm_pos unchanged |
| 2 | Move right | arm_pos + 1 (boundary: ≤ 4) |

In [ ]:
# ── Space dimensions ──────────────────────────────────────────────────────────
N_POS = 5        # arm positions: 0 (far left) … 4 (far right)
N_OBJ = 3        # object location: 0=left(pos1), 1=center(pos2), 2=right(pos3)

num_states  = [N_POS, N_OBJ]
num_obs     = [4, 2]    # visual: not_visible/left/center/right; tactile: none/contact
num_actions = [3]        # 0=left, 1=stay, 2=right

# Object location → physical arm position where contact occurs
OBJ_PHYS = {0: 1, 1: 2, 2: 3}

# Human-readable labels
pos_labels     = ['pos0', 'pos1', 'pos2', 'pos3', 'pos4']
obj_labels     = ['Left(pos1)', 'Center(pos2)', 'Right(pos3)']
vis_labels     = ['Not visible', 'See Left', 'See Center', 'See Right']
tact_labels    = ['No contact', 'Contact']
action_labels  = ['Move Left', 'Stay', 'Move Right']

print(f'State space:  arm_pos × obj_loc = {N_POS} × {N_OBJ} = {N_POS * N_OBJ} joint states')
print(f'Observations: visual({num_obs[0]}) + tactile({num_obs[1]})')
print(f'Actions:      {num_actions[0]}')

---
## 3. Generative Model: A, B, C, D Matrices

### A matrix design

**A_visual** `shape = (4, 5, 3)` — how visual observation depends on hidden state:
- Visual sees the *object location* regardless of arm position (external camera).
- Clean: obs=1 (left) ↔ obj_loc=0, obs=2 (center) ↔ obj_loc=1, obs=3 (right) ↔ obj_loc=2.
- **Occlusion = noise added to A_visual rows** → uniform distribution → observation becomes uninformative.

**A_tactile** `shape = (2, 5, 3)` — how tactile depends on hidden state:
- Contact (obs=1) when and only when `arm_pos == OBJ_PHYS[obj_loc]`.
- Always reliable — this modality is *not* degraded by visual occlusion.

In [ ]:
# ── A matrices ────────────────────────────────────────────────────────────────
A = utils.obj_array_zeros([[o] + num_states for o in num_obs])

# A_visual: obs depends on object location (same for all arm positions)
for arm_pos in range(N_POS):
    A[0][1, arm_pos, 0] = 1.0   # obj at left   → see 'left'
    A[0][2, arm_pos, 1] = 1.0   # obj at center → see 'center'
    A[0][3, arm_pos, 2] = 1.0   # obj at right  → see 'right'

# A_tactile: contact iff arm_pos matches object's physical position
for obj_loc in range(N_OBJ):
    phys = OBJ_PHYS[obj_loc]
    for arm_pos in range(N_POS):
        if arm_pos == phys:
            A[1][1, arm_pos, obj_loc] = 1.0   # contact
        else:
            A[1][0, arm_pos, obj_loc] = 1.0   # no contact

# Verify normalization
for m, name in enumerate(['visual', 'tactile']):
    ok = np.allclose(A[m].sum(axis=0), 1.0)
    print(f'A[{m}] ({name}) normalized: {ok}  shape={A[m].shape}')

# ── Visualize A matrices ──────────────────────────────────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(13, 4))

# A_visual: show slice at arm_pos=0 (identical for all arm positions)
ax = axes[0]
im = ax.imshow(A[0][:, 0, :], cmap='Blues', vmin=0, vmax=1, aspect='auto')
ax.set_xlabel('Object location'); ax.set_ylabel('Visual observation')
ax.set_xticks(range(N_OBJ)); ax.set_xticklabels(obj_labels, fontsize=8)
ax.set_yticks(range(num_obs[0])); ax.set_yticklabels(vis_labels, fontsize=8)
ax.set_title('A_visual  p(obs_visual | obj_loc)\n(same for all arm positions)')
plt.colorbar(im, ax=ax)
for i in range(num_obs[0]):
    for j in range(N_OBJ):
        v = A[0][i, 0, j]
        if v > 0: ax.text(j, i, f'{v:.1f}', ha='center', va='center', fontsize=9, color='white')

# A_tactile: show summed over obj_loc for clarity
# Reshape to (2, 5*3) for display
A1_flat = A[1].reshape(2, N_POS * N_OBJ)
ax = axes[1]
im = ax.imshow(A1_flat, cmap='Reds', vmin=0, vmax=1, aspect='auto')
ax.set_xlabel('(arm_pos, obj_loc) pairs — 15 combinations')
ax.set_ylabel('Tactile observation')
ax.set_yticks(range(2)); ax.set_yticklabels(tact_labels)
ax.set_title('A_tactile  p(obs_tactile | arm_pos, obj_loc)')
plt.colorbar(im, ax=ax)
# Annotate contact cells
for j in range(N_POS * N_OBJ):
    for i in range(2):
        v = A1_flat[i, j]
        if v > 0:
            color = 'white' if v > 0.5 else 'black'
            ax.text(j, i, f'{v:.1f}', ha='center', va='center', fontsize=7, color=color)

plt.suptitle('Generative model: observation matrices A', fontsize=12)
plt.tight_layout()
plt.savefig('a_matrices.png', dpi=100, bbox_inches='tight')
plt.show()
print('Saved: a_matrices.png')

In [ ]:
# ── B matrices: state transitions ─────────────────────────────────────────────
B = utils.obj_array(2)

# B[0]: arm position transitions  shape=(5,5,3)
B[0] = np.zeros((N_POS, N_POS, num_actions[0]))
for p in range(N_POS):
    B[0][max(0, p-1), p, 0] = 1.0          # action 0: move left
    B[0][p,           p, 1] = 1.0          # action 1: stay
    B[0][min(N_POS-1, p+1), p, 2] = 1.0   # action 2: move right

# B[1]: object location is fixed (unaffected by arm actions)  shape=(3,3,3)
B[1] = np.zeros((N_OBJ, N_OBJ, num_actions[0]))
for a in range(num_actions[0]):
    B[1][:, :, a] = np.eye(N_OBJ)

# ── C matrix: preferred observations ─────────────────────────────────────────
C = utils.obj_array_zeros(num_obs)
C[0] = np.zeros(num_obs[0])         # no preference over visual obs
C[1] = np.array([-1.0, 3.0])        # strongly prefer tactile contact (index 1)

# ── D matrix: prior beliefs at t=0 ───────────────────────────────────────────
D = utils.obj_array_uniform(num_states)
D[0] = np.array([1.0, 0.0, 0.0, 0.0, 0.0])   # arm starts at pos0 (certain)
D[1] = np.array([1/3, 1/3, 1/3])              # object location: completely unknown

print('B[0] (arm transitions)     shape:', B[0].shape)
print('B[1] (obj location fixed)  shape:', B[1].shape)
print('C[0] (visual pref)  :', C[0])
print('C[1] (tactile pref) :', C[1])
print('D[0] (arm prior)    :', D[0])
print('D[1] (obj prior)    :', D[1])

---
## 4. Precision Switching: Occlusion → A_visual Degrades

Occlusion is modelled by **injecting noise into A_visual**:

$$A_{visual}^{noisy} = (1 - \text{noise}) \cdot A_{visual}^{clean} + \text{noise} \cdot \frac{1}{N_{obs}}$$

- **noise = 0.0** (c_visual = 1.0): perfect visual — A_visual is identity-like
- **noise = 1.0** (c_visual = 0.0): complete occlusion — A_visual is uniform (no information)

This is equivalent to setting $\Pi_{visual} \propto (1 - \text{noise})$.

In [ ]:
def add_noise_to_A(A_clean, modality_idx, noise_level):
    """
    Degrade precision of modality m by mixing with uniform distribution.
    noise_level=0.0: clean (Pi=max); noise_level=1.0: uniform (Pi=0, no info)
    """
    A_noisy = copy.deepcopy(A_clean)
    n_obs = A_clean[modality_idx].shape[0]
    uniform = np.ones_like(A_clean[modality_idx]) / n_obs
    A_noisy[modality_idx] = (1 - noise_level) * A_clean[modality_idx] + noise_level * uniform
    return A_noisy

# Visualize A_visual under 3 noise levels
noise_levels = [0.0, 0.5, 1.0]
c_visual_approx = [1.0, 0.5, 0.0]

fig, axes = plt.subplots(1, 3, figsize=(14, 3.5))
for i, (noise, cv) in enumerate(zip(noise_levels, c_visual_approx)):
    A_n = add_noise_to_A(A, 0, noise)
    ax = axes[i]
    im = ax.imshow(A_n[0][:, 0, :], cmap='Blues', vmin=0, vmax=1, aspect='auto')
    ax.set_title(f'A_visual  noise={noise}\n(c_visual ~ {cv})')
    ax.set_xlabel('Object location')
    ax.set_ylabel('Visual obs' if i == 0 else '')
    ax.set_xticks(range(N_OBJ)); ax.set_xticklabels(obj_labels, fontsize=7)
    ax.set_yticks(range(4)); ax.set_yticklabels(vis_labels, fontsize=7)
    for r in range(4):
        for c_ in range(N_OBJ):
            v = A_n[0][r, 0, c_]
            color = 'white' if v > 0.5 else 'black'
            ax.text(c_, r, f'{v:.2f}', ha='center', va='center', fontsize=8, color=color)
    plt.colorbar(im, ax=ax)

plt.suptitle('Effect of occlusion on A_visual (precision degradation)', fontsize=12)
plt.tight_layout()
plt.savefig('a_visual_noise.png', dpi=100, bbox_inches='tight')
plt.show()
print('Saved: a_visual_noise.png')
print()
print('noise=0.0: each column is a delta function   -> perfect information')
print('noise=0.5: partial uncertainty               -> weak belief update')
print('noise=1.0: uniform across all rows           -> zero information')

---
## 5. Simulation: No Occlusion vs Complete Occlusion

We run two agents under the same task (object hidden at **right=pos3**) but different visual precision:

| Condition | A_visual | Expected behaviour |
|---|---|---|
| **No occlusion** | noise=0.0 | Visual instantly reveals object location → direct path |
| **Complete occlusion** | noise=1.0 | Visual is blind → agent explores until tactile contact resolves uncertainty |

### Key observation to watch
- **No occlusion**: `belief_obj` jumps to `[0, 0, 1]` at **t=0** (visual gives immediate certainty)
- **Complete occlusion**: `belief_obj` narrows step by step via **negative tactile evidence**
  (no contact at pos1 → not Left; no contact at pos2 → not Center; contact at pos3 → confirmed Right)

In [ ]:
TRUE_OBJ_LOC = 2   # object is at RIGHT (pos3) — the agent does not know this

def get_environment_observation(arm_pos, obj_loc):
    """
    True environment observation (perfect visual, perfect tactile).
    The agent's internal A matrix may be noisy — but the world is not.
    """
    visual  = obj_loc + 1             # 1=left, 2=center, 3=right
    tactile = 1 if arm_pos == OBJ_PHYS[obj_loc] else 0
    return [visual, tactile]

def run_simulation(A_agent, T=8, label=''):
    agent = Agent(A=A_agent, B=B, C=C, D=D, policy_len=2, inference_horizon=2)
    arm_pos = 0
    history = []

    for t in range(T):
        obs = get_environment_observation(arm_pos, TRUE_OBJ_LOC)
        qs         = agent.infer_states(obs)
        q_pi, G    = agent.infer_policies()
        action     = int(agent.sample_action()[0])

        history.append({
            't':         t,
            'arm_pos':   arm_pos,
            'obs':       obs,
            'belief_arm': qs[0].copy(),
            'belief_obj': qs[1].copy(),
            'action':    action,
            'G':         G,
        })

        arm_pos = max(0, min(N_POS - 1, arm_pos + (action - 1)))

    print(f'=== {label} ===')
    print(f'  (True object location: {obj_labels[TRUE_OBJ_LOC]})')
    header = f'{"t":>3}  {"arm":>4}  {"tactile":>8}  {"belief_obj (L/C/R)":>22}  action'
    print(header); print('-' * len(header))
    for h in history:
        ac   = action_labels[h['action']]
        tac  = 'CONTACT' if h['obs'][1] else 'none'
        bo   = h['belief_obj']
        print(f'{h["t"]:>3}  {h["arm_pos"]:>4}  {tac:>8}  '
              f'L={bo[0]:.2f} C={bo[1]:.2f} R={bo[2]:.2f}  {ac}')
    return history

log_clear = run_simulation(A, label='No occlusion  (c_visual = 1.0)')
print()
A_blind   = add_noise_to_A(A, 0, 1.0)
log_blind = run_simulation(A_blind, label='Complete occlusion  (c_visual = 0.0)')

In [ ]:
# ── Comparison visualization ───────────────────────────────────────────────────
T = len(log_clear)
steps = list(range(T))

fig, axes = plt.subplots(2, 3, figsize=(15, 7))

titles_row = ['No occlusion  (c_visual=1.0)', 'Complete occlusion  (c_visual=0.0)']
colors_obj = ['steelblue', 'darkorange', 'firebrick']

for row, (log, title) in enumerate(zip([log_clear, log_blind], titles_row)):

    # Panel 1: belief over object location
    ax = axes[row][0]
    for loc_i, (lbl, col) in enumerate(zip(obj_labels, colors_obj)):
        vals = [h['belief_obj'][loc_i] for h in log]
        ax.plot(steps, vals, '-o', color=col, linewidth=2, label=lbl)
    ax.axhline(1/3, color='gray', linestyle='--', alpha=0.5, label='Uniform prior')
    ax.set_title(f'{title}\nBelief over object location')
    ax.set_xlabel('Time step'); ax.set_ylabel('Belief probability')
    ax.set_ylim([0, 1.05]); ax.legend(fontsize=7)

    # Panel 2: arm position trajectory
    ax = axes[row][1]
    arm_traj = [h['arm_pos'] for h in log]
    ax.plot(steps, arm_traj, 'k-o', linewidth=2, markersize=8)
    ax.axhline(OBJ_PHYS[TRUE_OBJ_LOC], color='firebrick', linestyle='--',
               linewidth=1.5, label=f'Object at pos{OBJ_PHYS[TRUE_OBJ_LOC]}')
    # Mark contact moments
    for h in log:
        if h['obs'][1] == 1:
            ax.scatter(h['t'], h['arm_pos'], s=200, zorder=5,
                       color='firebrick', marker='*')
    ax.set_title(f'{title}\nArm position trajectory')
    ax.set_xlabel('Time step'); ax.set_ylabel('Arm position')
    ax.set_yticks(range(N_POS)); ax.set_yticklabels(pos_labels)
    ax.set_ylim([-0.5, N_POS - 0.5]); ax.legend(fontsize=8)
    ax.annotate('* = contact', xy=(0.62, 0.05), xycoords='axes fraction', fontsize=8,
                color='firebrick')

    # Panel 3: tactile observation over time
    ax = axes[row][2]
    tact_vals = [h['obs'][1] for h in log]
    bar_colors = ['firebrick' if v else 'lightgray' for v in tact_vals]
    ax.bar(steps, tact_vals, color=bar_colors, edgecolor='gray')
    ax.set_title(f'{title}\nTactile observation')
    ax.set_xlabel('Time step'); ax.set_ylabel('Tactile obs')
    ax.set_yticks([0, 1]); ax.set_yticklabels(tact_labels)
    ax.set_ylim([0, 1.3])

plt.suptitle(
    'AIF grasp task: visual vs tactile-guided navigation\n'
    'Red star = first contact event',
    fontsize=13
)
plt.tight_layout()
plt.savefig('condition_comparison.png', dpi=100, bbox_inches='tight')
plt.show()
print('Saved: condition_comparison.png')

In [ ]:
# ── Summary statistics ────────────────────────────────────────────────────────
def first_contact_step(log):
    for h in log:
        if h['obs'][1] == 1:
            return h['t']
    return None

def steps_to_certainty(log, threshold=0.95):
    """First step where belief over correct obj_loc > threshold."""
    for h in log:
        if h['belief_obj'][TRUE_OBJ_LOC] >= threshold:
            return h['t']
    return None

print('=== Summary ===')
print(f'Object location: {obj_labels[TRUE_OBJ_LOC]}')
print()
for label, log in [('No occlusion', log_clear), ('Complete occlusion', log_blind)]:
    fc  = first_contact_step(log)
    sc  = steps_to_certainty(log)
    print(f'{label}:')
    print(f'  First contact at step:         {fc}')
    print(f'  Belief certainty (>0.95) at:   {sc}')
    print(f'  Information source:            {"Visual (instant)" if label=="No occlusion" else "Tactile (search-then-confirm)"}')
    print()

---
## 6. Bridge to SO-101 Full System

### What this toy model demonstrates

| Demonstrated | SO-101 equivalent |
|---|---|
| Arm position factor (5 discrete) | 6-DoF joint angles, discretized per axis |
| Object location factor (3 discrete) | 3-D object pose estimate, clustered |
| A_visual degradation (noise injection) | `c_visual` score → entropy of visual feature map |
| A_tactile always reliable | AnySkin 15-ch magnetic sensor (phase 2) / servo current (phase 1) |
| Negative tactile evidence narrows belief | Johansson-Flanagan smart reflex — tactile as information |

### Differences to address in next notebooks

1. **Continuous joint angles** → discretization strategy per DoF (20 bins)
2. **6-DoF factored state space** → pymdp `A_factor_list` / `B_factor_list`
3. **Dynamic c_visual** → `add_noise_to_A` called per timestep based on live camera entropy
4. **AnySkin 15-ch** → contact pattern dictionary (16 patterns, learned offline)
5. **50 Hz real-time loop** → async inference thread + 50 Hz control thread

### Core equation (confirmed working in this notebook)

$$F_{total} = \Pi_{visual} \cdot \varepsilon_{visual}^2 + \Pi_{tactile} \cdot \varepsilon_{tactile}^2$$

When $c_{visual} \to 0$ (noise $\to$ 1.0):
- $\Pi_{visual} \to 0$ → visual prediction errors ignored
- Agent relies entirely on $\Pi_{tactile}$ → **contact search behaviour emerges automatically from EFE**

### Next notebook
`03_mujoco_env.ipynb` — MuJoCo environment setup for SO-101 (or simplified 2-DoF proxy)